# Colab Notebook: Download ETH Zurich Wheat Metadata & Processed Masks
This standalone notebook downloads the metadata and processed alignment toolkit archives for the ETH Zurich Wheat Leaf dataset (DOI `10.3929/ethz-b-000739194`), extracts them into temporary storage, packages `eth_wheat.zip`, and downloads it directly to your computer.

In [ ]:
# Step 1: Install required packages
!pip install -q requests tqdm pyyaml pydantic pandas

In [ ]:
# Step 2: Download ETH Wheat metadata & processed files with branch fallbacks
import os
import zipfile
import requests
from pathlib import Path

target_dir = Path('/content/data/raw/eth_wheat')
target_dir.mkdir(parents=True, exist_ok=True)

print('ETH Zurich Wheat Dataset Information:')
print('DOI: 10.3929/ethz-b-000739194')
print('Landing Page: https://doi.org/10.3929/ethz-b-000739194')

# Try master, main, and development branch URLs
urls = [
    'https://github.com/and-jonas/sympathique-wheat/archive/refs/heads/master.zip',
    'https://github.com/and-jonas/sympathique-wheat/archive/refs/heads/main.zip',
    'https://github.com/and-jonas/sympathique-wheat/archive/refs/heads/development.zip'
]

zip_path = target_dir / 'sympathique-wheat.zip'
success = False

for meta_url in urls:
    print(f'Fetching dataset metadata from {meta_url}...')
    response = requests.get(meta_url, stream=True)
    if response.status_code == 200:
        with open(zip_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print('Successfully downloaded metadata archive.')
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(target_dir)
        print(f'Extracted metadata files to {target_dir}')
        success = True
        break
    else:
        print(f'Branch URL returned HTTP status {response.status_code}. Trying next...')

if not success:
    print('Manual Portal Download URL: https://www.research-collection.ethz.ch/handle/10.3929/ethz-b-000739194')

In [ ]:
# Step 3: Zip dataset folder & trigger automatic download to your computer
from google.colab import files
import shutil

print('Compressing eth_wheat.zip...')
shutil.make_archive('/content/eth_wheat', 'zip', str(target_dir))
print('Triggering browser download for eth_wheat.zip...')
files.download('/content/eth_wheat.zip')